In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt,Command

In [21]:
class SimpleState(StateGraph):
    proposed_action:str
    approved:str
    result:str



In [22]:
def propose_action(state: SimpleState) -> dict:
    action = "Send a welcome email to new users"

    print(f"[propose_action] Proposing: '{action}'")

    return {"proposed_action": action}

def ask_human(state: SimpleState) -> dict:

    print(
        "[ask_human] Calling interrupt() "
        "-- graph will PAUSE now."
    )

    answer = interrupt({
        "question": (
            f"Approve this action? "
            f"'{state['proposed_action']}'"
        ),
        "hint": "Reply 'yes' to approve or 'no' to cancel."
    })

    approved = (
        str(answer).strip().lower() == "yes"
    )

    print(
        f"[ask_human] Resumed! "
        f"Human said: '{answer}' "
        f"--> approved={approved}"
    )

    return {
        "approved": approved
    }

def finish(state: SimpleState) -> dict:
    # Step 3: act on the human's decision
    if state["approved"]:
        result = f"ACTION EXECUTED: {state['proposed_action']}"
    else:
        result = "Action cancelled by human."

    print(f"[finish] {result}")

    return {"result": result}

In [ ]:
builder=StateGraph(SimpleState)
builder.add_node("proposed_action",propose_action)
builder.add_node("ask_question",ask_human)
builder.add_node("finish",finish)


builder.add_edge(START,"proposed_action")
builder.add_edge("proposed_action","ask_question")
builder.add_edge("ask_question","finish")
builder.add_edge("finish",END)


check_pointer=MemorySaver()
graph=builder.compile(checkpointer=check_pointer)

config={
    "configurable":{
        "thread_id":"demo-001"
    }
}

result=graph.invoke(
    {
    "proposed_action":"",
    "approved":False,
    "result":""
    },config
)


[propose_action] Proposing: 'Send a welcome email to new users'
[ask_human] Calling interrupt() -- graph will PAUSE now.
